# 同态倒谱初始化 + 最小熵反卷积（MED）提取 Click 多径结构

目标记录：`010 | Voyage_Ori_01 / PulseTrain_014 / Click_008`

本 Notebook 使用经典盲反卷积模型

$$y[n] = s[n] * h[n] + \varepsilon[n],$$

其中 $s[n]$ 是未知的短时 Click 源波形，$h[n]$ 是由直达径和反射径构成的稀疏多径响应。处理过程为：

1. 对原始波形进行 20–190 kHz 零相位带通预处理；
2. 对对数幅度谱做同态倒谱低倒频窗分离，重建最小相位源波形；
3. 由源波形计算正则化逆滤波器，作为 MED 的初值；
4. MED 迭代最大化反卷积输出峭度，恢复稀疏到达序列；
5. 从 MED 输出中检测显著到达，给出相对直达径的时延和强度，并保存图片与 CSV。

> 单通道、单条波形的盲反卷积存在尺度、符号和整体时移不确定性。因此输出用于解释**相对多径结构**，不把绝对相位或绝对幅值当作唯一物理解。


In [ ]:
from pathlib import Path
import csv
import json
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt, find_peaks, hilbert, windows, correlate, correlation_lags
from scipy.optimize import linear_sum_assignment

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, total=None, desc=None, **kwargs):
        print(desc or "处理中……")
        return iterable

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 125

PROJECT_ROOT = Path(r"D:\Project_Github\Indo-Pacific-humpback-dolphin")
INPUT_WAV = PROJECT_ROOT / r"00_Data\Positive_Clicks_3500\Voyage_Ori_01\PulseTrain_014\Click_008\Pulse_008.wav"
OUTPUT_DIR = PROJECT_ROOT / r"18_多径提取算法\Cepstrum_MED_010_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 可调参数
LOWCUT_HZ = 20_000.0
HIGHCUT_HZ = 190_000.0
FILTER_ORDER = 6
SOURCE_DURATION_MS = 0.22       # 倒谱低倒频窗及源波形长度
MED_FILTER_LENGTH_MS = 0.35     # MED 逆滤波器长度
MED_ITERATIONS = 80
MED_RIDGE = 1e-5
MIN_PATH_DISTANCE_MS = 0.035
PEAK_RELATIVE_THRESHOLD = 0.06
PEAK_PROMINENCE_RATIO = 0.025
MAX_PATHS = 12
BOOTSTRAP_RUNS = 200
BOOTSTRAP_NOISE_SCALE = 1.0       # 加入 1 倍静默段背景噪声
BOOTSTRAP_BLOCK_MS = 0.05         # 块重采样保留噪声的短时相关性
BOOTSTRAP_MATCH_TOLERANCE_MS = 0.04
BOOTSTRAP_RANDOM_SEED = 20260813

print(f"输入：{INPUT_WAV}")
print(f"输出：{OUTPUT_DIR}")
assert INPUT_WAV.exists(), f"找不到输入 WAV：{INPUT_WAV}"


## 1. 读取与预处理

保留完整 10 ms 波形，转为单声道浮点数，去直流后进行零相位带通滤波。Tukey 窗只用于轻微抑制截断边缘，避免倒谱把边界不连续误判为传播结构。


In [ ]:
fs, raw = wavfile.read(INPUT_WAV)
if raw.ndim == 2:
    raw = raw.mean(axis=1)

if np.issubdtype(raw.dtype, np.integer):
    scale = max(abs(np.iinfo(raw.dtype).min), np.iinfo(raw.dtype).max)
    y_raw = raw.astype(np.float64) / scale
else:
    y_raw = raw.astype(np.float64)

y_raw -= np.mean(y_raw)
nyquist = fs / 2.0
if not (0 < LOWCUT_HZ < HIGHCUT_HZ < nyquist):
    raise ValueError(f"带通范围必须低于 Nyquist 频率 {nyquist:,.0f} Hz")

sos = butter(
    FILTER_ORDER,
    [LOWCUT_HZ / nyquist, HIGHCUT_HZ / nyquist],
    btype="bandpass",
    output="sos",
)
y = sosfiltfilt(sos, y_raw)
y *= windows.tukey(len(y), alpha=0.02)
y /= np.max(np.abs(y)) + np.finfo(float).eps

t_ms = np.arange(len(y)) / fs * 1e3
freq = np.fft.rfftfreq(len(y), d=1 / fs) / 1e3
spectrum_db = 20 * np.log10(np.abs(np.fft.rfft(y)) / (np.max(np.abs(np.fft.rfft(y))) + 1e-12) + 1e-12)

print(f"采样率：{fs:,} Hz")
print(f"样本数：{len(y):,}（{len(y) / fs * 1e3:.3f} ms）")

fig, axes = plt.subplots(2, 1, figsize=(11, 6), constrained_layout=True)
axes[0].plot(t_ms, y_raw / (np.max(np.abs(y_raw)) + 1e-12), color="0.65", lw=0.8, label="原始")
axes[0].plot(t_ms, y, color="#176B87", lw=1.0, label="带通后")
axes[0].set(xlabel="时间 (ms)", ylabel="归一化幅值", title="原始与预处理 Click 波形")
axes[0].legend(loc="upper right")
axes[0].grid(alpha=0.2)
axes[1].plot(freq, spectrum_db, color="#B2533E", lw=1.0)
axes[1].axvspan(LOWCUT_HZ / 1e3, HIGHCUT_HZ / 1e3, color="#B2533E", alpha=0.08)
axes[1].set(xlim=(0, nyquist / 1e3), ylim=(-90, 3), xlabel="频率 (kHz)", ylabel="相对幅度 (dB)", title="预处理信号频谱")
axes[1].grid(alpha=0.2)
plt.show()


## 2. 同态倒谱初始化

实倒谱将卷积在对数谱域中转换为加法。低倒频成分主要描述平滑、短时的源波形谱包络；保留该部分并按最小相位规则重建源波形。随后求解

$$g_0 = \arg\min_g \|s_0 * g - \delta\|_2^2 + \lambda\|g\|_2^2,$$

得到 MED 的初始逆滤波器 $g_0$。这里的同态结果只承担初始化，不直接作为最终多径判定。


In [ ]:
def next_pow2(n):
    return 1 << int(np.ceil(np.log2(max(2, n))))


def homomorphic_source_initialization(signal, sample_rate, source_duration_ms):
    n = len(signal)
    nfft = next_pow2(4 * n)
    spectrum = np.fft.fft(signal, nfft)
    log_magnitude = np.log(np.maximum(np.abs(spectrum), 1e-10 * np.max(np.abs(spectrum))))
    real_cepstrum = np.fft.ifft(log_magnitude).real

    q_cut = max(3, int(round(source_duration_ms * 1e-3 * sample_rate)))
    q_cut = min(q_cut, nfft // 4)
    low_cepstrum = np.zeros(nfft)
    low_cepstrum[0] = real_cepstrum[0]
    low_cepstrum[1:q_cut + 1] = 2.0 * real_cepstrum[1:q_cut + 1]
    if nfft % 2 == 0:
        low_cepstrum[nfft // 2] = real_cepstrum[nfft // 2]

    minimum_phase_spectrum = np.exp(np.fft.fft(low_cepstrum))
    source_full = np.fft.ifft(minimum_phase_spectrum).real
    source_length = max(9, 2 * q_cut + 1)
    source = source_full[:source_length].copy()
    source *= windows.tukey(source_length, alpha=0.35)
    source -= np.mean(source)
    source /= np.linalg.norm(source) + 1e-12
    return source, real_cepstrum, low_cepstrum, q_cut, nfft


def regularized_inverse_filter(source, filter_length, ridge=1e-3):
    source = np.asarray(source, dtype=float)
    conv_length = len(source) + filter_length - 1
    conv_matrix = np.zeros((conv_length, filter_length))
    for column in range(filter_length):
        conv_matrix[column:column + len(source), column] = source

    target = np.zeros(conv_length)
    target[min(len(source) - 1, conv_length - 1)] = 1.0
    lhs = conv_matrix.T @ conv_matrix + ridge * np.eye(filter_length)
    rhs = conv_matrix.T @ target
    inverse_filter = np.linalg.solve(lhs, rhs)
    inverse_filter /= np.linalg.norm(inverse_filter) + 1e-12
    return inverse_filter


source0, cepstrum, cepstrum_low, q_cut, nfft = homomorphic_source_initialization(
    y, fs, SOURCE_DURATION_MS
)
med_filter_length = max(9, int(round(MED_FILTER_LENGTH_MS * 1e-3 * fs)))
g0 = regularized_inverse_filter(source0, med_filter_length, ridge=1e-3)

q_us = np.arange(nfft // 2) / fs * 1e6
source_t_us = np.arange(len(source0)) / fs * 1e6
print(f"倒谱低倒频截止：{q_cut} 点（{q_cut / fs * 1e6:.1f} μs）")
print(f"初始源波形：{len(source0)} 点；MED 滤波器：{len(g0)} 点")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
show_q = q_us <= 700
axes[0].plot(q_us[show_q], cepstrum[:nfft // 2][show_q], color="#5B5F97", lw=1.0)
axes[0].axvline(q_cut / fs * 1e6, color="#C73E1D", ls="--", label="低倒频截止")
axes[0].set(xlabel="倒频 (μs)", ylabel="实倒谱", title="实倒谱及源/多径分离截止")
axes[0].legend()
axes[0].grid(alpha=0.2)
axes[1].plot(source_t_us, source0, color="#176B87", lw=1.2)
axes[1].set(xlabel="时间 (μs)", ylabel="归一化幅值", title="同态倒谱重建的最小相位源波形")
axes[1].grid(alpha=0.2)
plt.show()


## 3. MED 优化

将信号的滞后样本组成矩阵 $X$，反卷积输出为 $z=Xg$。MED 通过固定点迭代更新逆滤波器，使输出的归一化四阶矩（峭度）增大：

$$g \leftarrow R_{xx}^{-1}X^Tz^3, \qquad R_{xx}=X^TX+\lambda I.$$

每次更新后进行能量归一化和阻尼，以提高有限长度、含噪 Click 上的稳定性。代码输出窗口中的“总体进度”覆盖全部 MED 迭代。


In [ ]:
def lag_matrix(signal, filter_length):
    n_rows = len(signal) - filter_length + 1
    if n_rows <= 0:
        raise ValueError("MED 滤波器长度必须小于信号长度")
    return np.stack(
        [signal[i:i + filter_length][::-1] for i in range(n_rows)],
        axis=0,
    )


def excess_kurtosis(values):
    values = values - np.mean(values)
    second = np.mean(values ** 2) + 1e-15
    return np.mean(values ** 4) / (second ** 2) - 3.0


def med_deconvolution(signal, initial_filter, iterations=80, ridge=1e-5, damping=0.35, show_progress=True):
    X = lag_matrix(signal, len(initial_filter))
    n_rows = X.shape[0]
    R = (X.T @ X) / n_rows
    regularization = ridge * max(np.trace(R) / len(initial_filter), 1e-12)
    R_reg = R + regularization * np.eye(len(initial_filter))

    g = initial_filter.astype(float).copy()
    g /= np.sqrt(g @ R_reg @ g) + 1e-12
    history = []

    iterator = tqdm(range(iterations), desc="MED 总体进度", unit="次") if show_progress else range(iterations)
    for _ in iterator:
        z = X @ g
        z -= np.mean(z)
        z_scale = np.sqrt(np.mean(z ** 2)) + 1e-12
        z_normalized = z / z_scale
        rhs = (X.T @ (z_normalized ** 3)) / n_rows
        candidate = np.linalg.solve(R_reg, rhs)
        candidate /= np.sqrt(candidate @ R_reg @ candidate) + 1e-12

        if np.dot(candidate, g) < 0:
            candidate = -candidate
        g = (1.0 - damping) * g + damping * candidate
        g /= np.sqrt(g @ R_reg @ g) + 1e-12

        output = X @ g
        history.append(excess_kurtosis(output))

    output = X @ g
    output -= np.median(output)
    return output, g, np.asarray(history), X


X0 = lag_matrix(y, len(g0))
z0 = X0 @ g0
z_med, g_med, kurtosis_history, X = med_deconvolution(
    y,
    g0,
    iterations=MED_ITERATIONS,
    ridge=MED_RIDGE,
)

output_t_ms = (np.arange(len(z_med)) + len(g_med) - 1) / fs * 1e3
z0 /= np.max(np.abs(z0)) + 1e-12
z_med /= np.max(np.abs(z_med)) + 1e-12

print(f"初始输出超额峭度：{excess_kurtosis(z0):.3f}")
print(f"MED 输出超额峭度：{excess_kurtosis(z_med):.3f}")
print(f"最终一次迭代记录：{kurtosis_history[-1]:.3f}")

fig, ax = plt.subplots(figsize=(10.5, 3.6))
ax.plot(np.arange(1, len(kurtosis_history) + 1), kurtosis_history, color="#C73E1D", lw=1.5)
ax.set(xlabel="MED 迭代次数", ylabel="输出超额峭度", title="MED 收敛曲线")
ax.grid(alpha=0.22)
plt.show()


## 4. 多径到达检测

对 MED 输出的解析包络进行轻微平滑，并根据相对峰值、稳健噪声门限、突出度和最小到达间隔筛选候选。最早的显著候选定义为直达径；其他候选均相对于它报告时延与强度。


In [ ]:
def moving_average(values, length):
    length = max(1, int(length))
    kernel = np.ones(length) / length
    return np.convolve(values, kernel, mode="same")


envelope = np.abs(hilbert(z_med))
envelope = moving_average(envelope, max(3, int(round(0.008e-3 * fs))))
envelope /= np.max(envelope) + 1e-12

median = np.median(envelope)
mad = np.median(np.abs(envelope - median)) + 1e-12
height_threshold = max(PEAK_RELATIVE_THRESHOLD, median + 7.0 * 1.4826 * mad)
distance_samples = max(1, int(round(MIN_PATH_DISTANCE_MS * 1e-3 * fs)))

peak_indices, properties = find_peaks(
    envelope,
    height=height_threshold,
    prominence=PEAK_PROMINENCE_RATIO,
    distance=distance_samples,
)

if len(peak_indices) == 0:
    peak_indices = np.asarray([int(np.argmax(envelope))])
    properties = {"peak_heights": envelope[peak_indices], "prominences": envelope[peak_indices]}

# 保留最强的有限个候选，然后恢复时间顺序
if len(peak_indices) > MAX_PATHS:
    strongest = np.argsort(envelope[peak_indices])[-MAX_PATHS:]
    peak_indices = peak_indices[strongest]
peak_indices = np.sort(peak_indices)

direct_index = int(peak_indices[0])
direct_amplitude = float(envelope[direct_index])
paths = []
for path_number, index in enumerate(peak_indices):
    amplitude = float(envelope[index])
    relative_delay_ms = (int(index) - direct_index) / fs * 1e3
    relative_level_db = 20.0 * np.log10((amplitude + 1e-12) / (direct_amplitude + 1e-12))
    paths.append({
        "path": "direct" if path_number == 0 else f"multipath_{path_number}",
        "sample_index_med": int(index),
        "arrival_time_ms": float(output_t_ms[index]),
        "relative_delay_ms": float(relative_delay_ms),
        "relative_amplitude": amplitude / (direct_amplitude + 1e-12),
        "relative_level_db": float(relative_level_db),
    })

print(f"检测门限：{height_threshold:.4f}；检测到 {len(paths)} 条显著到达")
print(f"{'路径':<14}{'到达时刻(ms)':>14}{'相对时延(ms)':>14}{'相对强度(dB)':>15}")
for item in paths:
    print(
        f"{item['path']:<14}{item['arrival_time_ms']:>14.4f}"
        f"{item['relative_delay_ms']:>14.4f}{item['relative_level_db']:>15.2f}"
    )


## 5. Bootstrap 多径稳定性置信度

从直达径前的静默段估计背景噪声，并采用移动块 Bootstrap 保留噪声的短时相关性。每次将重采样噪声加入观测波形，完整重跑同态倒谱初始化、MED 和峰值检测。以原结果的直达径为锚点，各路径在相对时延容差内被重新检出的比例定义为

$$C_i=\frac{N_{\mathrm{match},i}}{N_{\mathrm{bootstrap}}}\times100\%.$$

这里的置信度表示对背景噪声扰动的**检测稳定性**，不是路径真实存在的贝叶斯后验概率。


In [ ]:
def detect_arrival_peaks(response, sample_rate):
    response_envelope = np.abs(hilbert(response))
    response_envelope = moving_average(response_envelope, max(3, int(round(0.008e-3 * sample_rate))))
    response_envelope /= np.max(response_envelope) + 1e-12
    center = np.median(response_envelope)
    spread = np.median(np.abs(response_envelope - center)) + 1e-12
    threshold = max(PEAK_RELATIVE_THRESHOLD, center + 7.0 * 1.4826 * spread)
    indices, _ = find_peaks(
        response_envelope,
        height=threshold,
        prominence=PEAK_PROMINENCE_RATIO,
        distance=max(1, int(round(MIN_PATH_DISTANCE_MS * 1e-3 * sample_rate))),
    )
    if len(indices) > MAX_PATHS:
        indices = indices[np.argsort(response_envelope[indices])[-MAX_PATHS:]]
    return np.sort(indices), response_envelope


def moving_block_resample(noise, output_length, block_length, rng):
    noise = np.asarray(noise, dtype=float) - np.mean(noise)
    block_length = min(max(2, int(block_length)), len(noise))
    maximum_start = len(noise) - block_length
    blocks_needed = int(np.ceil(output_length / block_length))
    starts = rng.integers(0, maximum_start + 1, size=blocks_needed)
    sampled = np.concatenate([noise[start:start + block_length] for start in starts])[:output_length]
    return sampled - np.mean(sampled)


# 静默段截止在首个显著到达前 0.5 ms，避免把 Click 前振铃当作背景噪声。
nominal_direct_signal_index = direct_index + len(g_med) - 1
guard_samples = int(round(0.5e-3 * fs))
noise_end = max(128, nominal_direct_signal_index - guard_samples)
background_noise = y[:noise_end].copy()
if len(background_noise) < 128:
    raise ValueError("直达径前静默段过短，无法进行可靠 Bootstrap")

reference_delays_ms = np.asarray([item["relative_delay_ms"] for item in paths])
reference_direct_index = direct_index
match_tolerance_samples = max(1, int(round(BOOTSTRAP_MATCH_TOLERANCE_MS * 1e-3 * fs)))
block_samples = max(2, int(round(BOOTSTRAP_BLOCK_MS * 1e-3 * fs)))
rng = np.random.default_rng(BOOTSTRAP_RANDOM_SEED)
match_counts = np.zeros(len(paths), dtype=int)
matched_delays = [[] for _ in paths]

for _ in tqdm(range(BOOTSTRAP_RUNS), desc="Bootstrap 置信度总体进度", unit="次"):
    perturbation = moving_block_resample(background_noise, len(y), block_samples, rng)
    y_bootstrap = y + BOOTSTRAP_NOISE_SCALE * perturbation
    y_bootstrap /= np.max(np.abs(y_bootstrap)) + 1e-12

    source_bootstrap, _, _, _, _ = homomorphic_source_initialization(y_bootstrap, fs, SOURCE_DURATION_MS)
    inverse_bootstrap = regularized_inverse_filter(source_bootstrap, med_filter_length, ridge=1e-3)
    response_bootstrap, _, _, _ = med_deconvolution(
        y_bootstrap, inverse_bootstrap, iterations=MED_ITERATIONS, ridge=MED_RIDGE, show_progress=False
    )
    response_bootstrap /= np.max(np.abs(response_bootstrap)) + 1e-12
    candidate_indices, _ = detect_arrival_peaks(response_bootstrap, fs)
    if len(candidate_indices) == 0:
        continue

    # 先匹配名义直达径，再以该 Bootstrap 直达径为零时延。
    direct_distances = np.abs(candidate_indices - reference_direct_index)
    direct_position = int(np.argmin(direct_distances))
    if direct_distances[direct_position] > match_tolerance_samples:
        continue
    bootstrap_direct_index = int(candidate_indices[direct_position])
    candidate_delays_ms = (candidate_indices - bootstrap_direct_index) / fs * 1e3

    cost = np.abs(reference_delays_ms[:, None] - candidate_delays_ms[None, :])
    reference_rows, candidate_columns = linear_sum_assignment(cost)
    for reference_row, candidate_column in zip(reference_rows, candidate_columns):
        if cost[reference_row, candidate_column] <= BOOTSTRAP_MATCH_TOLERANCE_MS:
            match_counts[reference_row] += 1
            matched_delays[reference_row].append(float(candidate_delays_ms[candidate_column]))

for index, item in enumerate(paths):
    confidence = 100.0 * match_counts[index] / BOOTSTRAP_RUNS
    samples = np.asarray(matched_delays[index], dtype=float)
    item["confidence_percent"] = float(confidence)
    item["bootstrap_matches"] = int(match_counts[index])
    item["bootstrap_runs"] = int(BOOTSTRAP_RUNS)
    item["bootstrap_delay_mean_ms"] = float(np.mean(samples)) if len(samples) else None
    item["bootstrap_delay_std_ms"] = float(np.std(samples, ddof=1)) if len(samples) > 1 else 0.0 if len(samples) else None

print(f"背景噪声段：0–{noise_end / fs * 1e3:.3f} ms；匹配容差：±{BOOTSTRAP_MATCH_TOLERANCE_MS:.3f} ms")
print(f"{'路径':<14}{'相对时延(ms)':>14}{'匹配次数':>12}{'置信度':>12}{'时延标准差(ms)':>18}")
for item in paths:
    delay_std = item['bootstrap_delay_std_ms']
    delay_std_text = f"{delay_std:.4f}" if delay_std is not None else "NA"
    print(
        f"{item['path']:<14}{item['relative_delay_ms']:>14.4f}"
        f"{item['bootstrap_matches']:>8}/{item['bootstrap_runs']:<3}"
        f"{item['confidence_percent']:>11.1f}%{delay_std_text:>18}"
    )


## 6. 波形重构与解释率

用倒谱估计的源波形与 MED 到达响应卷积得到重构波形：

$$\hat y_0 = \hat s * \hat h.$$

由于单通道盲反卷积存在整体时移、尺度和直流偏置不确定性，本节先用互相关估计整体时延，再通过最小二乘拟合增益与偏置。解释率在完整 10 ms 预处理波形上按

$$R^2=1-\frac{\sum_n(y[n]-\hat y[n])^2}{\sum_n(y[n]-\bar y)^2}$$

计算。同时报告完整 MED 响应和仅保留自动检测到达峰的稀疏响应，后者更直接反映图中多径结构能够解释的波形比例。


In [ ]:
def align_scale_and_score(observed, candidate):
    observed = np.asarray(observed, dtype=float)
    candidate = np.asarray(candidate, dtype=float)
    centered_observed = observed - np.mean(observed)
    centered_candidate = candidate - np.mean(candidate)
    cross_correlation = correlate(centered_observed, centered_candidate, mode="full", method="fft")
    lags = correlation_lags(len(observed), len(candidate), mode="full")
    lag_samples = int(lags[np.argmax(np.abs(cross_correlation))])

    aligned = np.zeros_like(observed)
    if lag_samples >= 0:
        count = min(len(candidate), len(observed) - lag_samples)
        if count > 0:
            aligned[lag_samples:lag_samples + count] = candidate[:count]
    else:
        start = -lag_samples
        count = min(len(candidate) - start, len(observed))
        if count > 0:
            aligned[:count] = candidate[start:start + count]

    design = np.column_stack([aligned, np.ones(len(aligned))])
    gain, offset = np.linalg.lstsq(design, observed, rcond=None)[0]
    fitted = gain * aligned + offset
    residual = observed - fitted
    sse = float(np.sum(residual ** 2))
    sst = float(np.sum((observed - np.mean(observed)) ** 2))
    r_squared = 1.0 - sse / (sst + 1e-15)
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    return fitted, r_squared, rmse, lag_samples, float(gain), float(offset)


# 完整 MED 响应重构
full_response = z_med.copy()
full_candidate = np.convolve(source0, full_response, mode="full")
y_hat_full, r2_full, rmse_full, lag_full, gain_full, offset_full = align_scale_and_score(y, full_candidate)

# 仅保留图中检测到的到达；在每个包络峰附近取实际带符号尖峰
sparse_response = np.zeros_like(z_med)
local_radius = max(1, int(round(0.010e-3 * fs)))
sparse_impulse_indices = []
for envelope_index in peak_indices:
    left = max(0, int(envelope_index) - local_radius)
    right = min(len(z_med), int(envelope_index) + local_radius + 1)
    impulse_index = left + int(np.argmax(np.abs(z_med[left:right])))
    sparse_response[impulse_index] = z_med[impulse_index]
    sparse_impulse_indices.append(impulse_index)
sparse_candidate = np.convolve(source0, sparse_response, mode="full")
y_hat_sparse, r2_sparse, rmse_sparse, lag_sparse, gain_sparse, offset_sparse = align_scale_and_score(y, sparse_candidate)

print(f"完整 MED 响应重构解释率 R^2：{r2_full:.4f}（{100 * r2_full:.2f}%）")
print(f"检测多径峰稀疏重构解释率 R^2：{r2_sparse:.4f}（{100 * r2_sparse:.2f}%）")
print(f"完整响应对齐：{lag_full} 点（{lag_full / fs * 1e3:.4f} ms），增益：{gain_full:.5g}，RMSE：{rmse_full:.5f}")
print(f"稀疏响应对齐：{lag_sparse} 点（{lag_sparse / fs * 1e3:.4f} ms），增益：{gain_sparse:.5g}，RMSE：{rmse_sparse:.5f}")

reconstruction_fig, reconstruction_axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
reconstruction_axes[0].plot(t_ms, y, color="0.25", lw=1.0, label="观测波形")
reconstruction_axes[0].plot(t_ms, y_hat_full, color="#176B87", lw=0.9, alpha=0.9, label=f"完整 MED 重构，R^2={r2_full:.3f}")
reconstruction_axes[0].set(title="完整 MED 响应的波形重构", ylabel="归一化幅值")
reconstruction_axes[0].legend(loc="upper right")
reconstruction_axes[0].grid(alpha=0.18)
reconstruction_axes[1].plot(t_ms, y, color="0.25", lw=1.0, label="观测波形")
reconstruction_axes[1].plot(t_ms, y_hat_sparse, color="#C73E1D", lw=0.9, alpha=0.9, label=f"检测峰稀疏重构，R^2={r2_sparse:.3f}")
reconstruction_axes[1].set(title="自动检测多径峰的稀疏波形重构", xlabel="时间 (ms)", ylabel="归一化幅值")
reconstruction_axes[1].legend(loc="upper right")
reconstruction_axes[1].grid(alpha=0.18)
reconstruction_path = OUTPUT_DIR / "cepstrum_med_reconstruction.png"
reconstruction_fig.savefig(reconstruction_path, dpi=220, bbox_inches="tight")
plt.show()

# 单独绘制估计的无多径原始 Click（同态倒谱最小相位源估计）
estimated_click = source0 / (np.max(np.abs(source0)) + 1e-12)
estimated_click_envelope = np.abs(hilbert(estimated_click))
estimated_click_path = OUTPUT_DIR / "estimated_original_click.png"

source_fig, source_ax = plt.subplots(figsize=(10.5, 4.5), constrained_layout=True)
source_ax.plot(source_t_us, estimated_click, color="#176B87", lw=1.25, label="估计的原始 Click")
source_ax.plot(source_t_us, estimated_click_envelope, color="#C73E1D", lw=1.0, alpha=0.8, label="解析包络")
source_ax.plot(source_t_us, -estimated_click_envelope, color="#C73E1D", lw=1.0, alpha=0.8)
source_ax.axhline(0, color="0.45", lw=0.8, ls=":")
source_ax.set(
    xlabel="相对时间 (μs)",
    ylabel="归一化幅值",
    title="估计的原始无多径 Click 信号（同态倒谱最小相位源估计）",
)
source_ax.set_xlim(source_t_us[0], source_t_us[-1])
source_ax.grid(alpha=0.22)
source_ax.legend(loc="upper right")
source_fig.savefig(estimated_click_path, dpi=220, bbox_inches="tight")
plt.show()
print(f"估计的原始 Click 图：{estimated_click_path}")


## 7. 综合结果图与文件导出

综合图同时展示预处理波形、同态初始化输出、MED 稀疏响应及最终多径时延结构。红色虚线标记自动检测的显著到达，绿色表示直达径。


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 12), constrained_layout=True)

axes[0].plot(t_ms, y, color="#176B87", lw=0.9)
axes[0].set(ylabel="幅值", title="(a) 带通预处理后的观测 Click")
axes[0].grid(alpha=0.18)

axes[1].plot(output_t_ms, z0, color="#7A5195", lw=0.9)
axes[1].set(ylabel="归一化幅值", title="(b) 同态倒谱初始化的反卷积输出")
axes[1].grid(alpha=0.18)

axes[2].plot(output_t_ms, z_med, color="0.55", lw=0.7, label="MED 输出")
axes[2].plot(output_t_ms, envelope, color="#C73E1D", lw=1.2, label="平滑包络")
axes[2].axhline(height_threshold, color="#C73E1D", ls=":", lw=1.0, label="检测门限")
for i, index in enumerate(peak_indices):
    color = "#188977" if i == 0 else "#C73E1D"
    axes[2].axvline(output_t_ms[index], color=color, ls="--", lw=0.9, alpha=0.8)
    axes[2].scatter(output_t_ms[index], envelope[index], color=color, s=32, zorder=4)
axes[2].set(ylabel="归一化幅值", title="(c) MED 稀疏到达响应与自动检测")
axes[2].legend(loc="upper right", ncol=3, fontsize=8)
axes[2].grid(alpha=0.18)

delays = np.asarray([item["relative_delay_ms"] for item in paths])
levels = np.asarray([item["relative_level_db"] for item in paths])
markerline, stemlines, baseline = axes[3].stem(delays, levels, linefmt="#C73E1D", markerfmt="o", basefmt=" ")
plt.setp(markerline, markersize=7, markerfacecolor="#C73E1D", markeredgecolor="white")
plt.setp(stemlines, linewidth=2, alpha=0.75)
if len(paths):
    axes[3].scatter(delays[0], levels[0], s=80, color="#188977", edgecolor="white", zorder=5)
for i, item in enumerate(paths):
    label = ("直达径" if i == 0 else f"多径 {i}\n{item['relative_delay_ms']:.3f} ms") + f"\n置信度 {item['confidence_percent']:.1f}%"
    axes[3].annotate(
        label,
        (delays[i], levels[i]),
        xytext=(5, 7),
        textcoords="offset points",
        fontsize=8,
    )
axes[3].axhline(0, color="0.45", ls=":", lw=0.9)
axes[3].set(
    xlabel="相对直达径时延 (ms)",
    ylabel="相对直达径强度 (dB)",
    title="(d) 010 | Voyage_Ori_01 / PulseTrain_014 / Click_008 多径结构",
)
axes[3].grid(alpha=0.2)
axes[3].margins(y=0.18)

for ax in axes[:3]:
    ax.set_xlim(t_ms[0], t_ms[-1])
    ax.set_xlabel("时间 (ms)")

figure_path = OUTPUT_DIR / "cepstrum_med_multipath_overview.png"
fig.savefig(figure_path, dpi=220, bbox_inches="tight")
plt.show()

structure_fig, structure_ax = plt.subplots(figsize=(10.5, 4.5), constrained_layout=True)
markerline, stemlines, baseline = structure_ax.stem(delays, levels, linefmt="#B32134", markerfmt="o", basefmt=" ")
plt.setp(markerline, markersize=8, markerfacecolor="#B32134", markeredgecolor="white")
plt.setp(stemlines, linewidth=2.2, alpha=0.8)
structure_ax.scatter(delays[0], levels[0], s=90, color="#188977", edgecolor="white", zorder=5)
for i, item in enumerate(paths):
    text = ("直达径" if i == 0 else f"多径 {i}\n{item['relative_delay_ms']:.3f} ms, {item['relative_level_db']:.1f} dB") + f"\n置信度 {item['confidence_percent']:.1f}%"
    structure_ax.annotate(text, (delays[i], levels[i]), xytext=(6, 8), textcoords="offset points", fontsize=9)
structure_ax.axhline(0, color="0.45", ls=":", lw=0.9)
structure_ax.set(
    xlabel="相对直达径时延 (ms)",
    ylabel="相对直达径强度 (dB)",
    title="同态倒谱初始化 + MED：多径时延结构",
)
structure_ax.grid(alpha=0.22)
structure_ax.margins(y=0.18)
structure_path = OUTPUT_DIR / "cepstrum_med_path_structure.png"
structure_fig.savefig(structure_path, dpi=220, bbox_inches="tight")
plt.show()

csv_path = OUTPUT_DIR / "cepstrum_med_paths.csv"
with csv_path.open("w", newline="", encoding="utf-8-sig") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(paths[0].keys()))
    writer.writeheader()
    writer.writerows(paths)

metadata = {
    "input_wav": str(INPUT_WAV),
    "sample_rate_hz": int(fs),
    "method": "homomorphic cepstrum initialization + minimum entropy deconvolution (MED)",
    "parameters": {
        "bandpass_hz": [LOWCUT_HZ, HIGHCUT_HZ],
        "source_duration_ms": SOURCE_DURATION_MS,
        "med_filter_length_ms": MED_FILTER_LENGTH_MS,
        "med_iterations": MED_ITERATIONS,
        "med_ridge": MED_RIDGE,
        "minimum_path_distance_ms": MIN_PATH_DISTANCE_MS,
        "peak_relative_threshold": PEAK_RELATIVE_THRESHOLD,
        "peak_prominence_ratio": PEAK_PROMINENCE_RATIO,
        "bootstrap_runs": BOOTSTRAP_RUNS,
        "bootstrap_noise_scale": BOOTSTRAP_NOISE_SCALE,
        "bootstrap_block_ms": BOOTSTRAP_BLOCK_MS,
        "bootstrap_match_tolerance_ms": BOOTSTRAP_MATCH_TOLERANCE_MS,
        "bootstrap_random_seed": BOOTSTRAP_RANDOM_SEED,
    },
    "initial_excess_kurtosis": float(excess_kurtosis(z0)),
    "med_excess_kurtosis": float(excess_kurtosis(z_med)),
    "reconstruction": {
        "full_med_response_r_squared": float(r2_full),
        "detected_peaks_sparse_r_squared": float(r2_sparse),
        "full_med_response_rmse": float(rmse_full),
        "detected_peaks_sparse_rmse": float(rmse_sparse),
        "full_alignment_lag_samples": int(lag_full),
        "sparse_alignment_lag_samples": int(lag_sparse),
        "full_fitted_gain": float(gain_full),
        "sparse_fitted_gain": float(gain_sparse),
    },
    "paths": paths,
}
json_path = OUTPUT_DIR / "cepstrum_med_metadata.json"
json_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"重构对比图：{reconstruction_path}")
print(f"综合图：{figure_path}")
print(f"多径结构图：{structure_path}")
print(f"路径表：{csv_path}")
print(f"参数与结果：{json_path}")


## 8. 结果解释注意事项

- `0 ms` 表示自动识别出的最早显著到达，并非 WAV 文件的绝对起点。
- 相对强度以该直达径为 0 dB；正值表示某个后续到达在 MED 包络中强于最早到达，这在几何聚焦或最早径较弱时可能出现。
- 倒谱截止、MED 滤波器长度和峰值门限都会影响分辨率；如果用于批量统计，应在人工标注集上统一标定参数。
- MED 强调稀疏性，可能把近距离重叠到达合并，也可能把源波形残余旁瓣当作弱候选。因此最终图适合用于结构检查，重要结论仍应结合原波形、相关性或传播模型验证。
- 重构解释率经过整体时延、增益与偏置拟合，是样本内描述性指标，不代表算法在其他 Click 上的泛化性能；稀疏峰重构解释率更适合评价已展示的多径结构。
- Bootstrap 置信度只衡量当前噪声模型和参数下的重检稳定性，不等于多径真实存在的概率；背景噪声非平稳时应分层重采样或采用独立重复 Click 验证。
